# Bypass R1 Oscillation Analysis — ASC vs DESC vs Combined

This notebook investigates the **signal oscillations** observed on the Antwerp
bypass (R1) near the Oosterweel construction works.

## Hypothesis

SAR backscatter depends strongly on the **viewing geometry** (incidence angle,
look direction). Since ascending and descending passes illuminate structures
from different sides, mixing them can introduce apparent oscillations in the
time-series that are purely geometrical — not physical change.

## Approach

1. Download OPERA RTC-S1 monthly passes over the bypass area
2. Split into **ascending-only**, **descending-only**, and **combined** subsets
3. Compare backscatter time-series at key points on the bypass
4. If ASC-only and DESC-only are smoother → oscillation is a geometry artefact
5. Generate separate GIFs for visual comparison

In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pyproj

from rs_tools.config import BoundingBox, SearchConfig
from rs_tools.search import search_archive
from rs_tools.datasets.catalog import get as get_dataset
from rs_tools.datasets.coverage import (
    summarize_search_results,
    print_coverage_report,
    filter_by_coverage,
    records_to_items,
)
from rs_tools.datasets.loader import (
    load_items,
    load_passes_from_disk,
    setup_terrascope_auth,
    LoadedItem,
)
from rs_tools.visualization.rtc_composite import rtc_composite
from rs_tools.visualization.scalebar import add_scalebar
from rs_tools.visualization.animation import save_timeseries_gif_lazy
from rs_tools.visualization.overlays import fetch_roads, overlay_roads, annotate_location

## 1. Define AOI & Working Directory

We use a tighter bounding box focused on the R1 bypass and Oosterweel junction.

In [ ]:
# Antwerp bypass region — focused on R1 bypass and junction
bbox_bypass = BoundingBox(west=4.34, south=51.19, east=4.46, north=51.27)

# Working directory
WORKDIR = os.path.join(os.getcwd(), "output")
GIF_DIR = os.path.join(WORKDIR, "gifs")
os.makedirs(GIF_DIR, exist_ok=True)
os.makedirs(os.path.join(WORKDIR, "passes"), exist_ok=True)

# Temporal range — full record, monthly sampling
START_DATE = "2014-10-01"
END_DATE   = "2026-03-24"
INTERVAL_MONTHS = 1   # monthly for better oscillation characterisation

# Key points on the bypass for time-series extraction
BYPASS_POINTS = {
    "Bypass R1 north":     (4.4050, 51.2500),
    "Bypass R1 mid":       (4.4100, 51.2350),
    "Bypass R1 south":     (4.4050, 51.2200),
    "Scheldt crossing":    (4.3885, 51.2210),
    "Tunnel north":        (4.3925, 51.2340),
}

print(f"AOI: {bbox_bypass}")
print(f"Period: {START_DATE} → {END_DATE} (monthly)")
print(f"Points of interest: {len(BYPASS_POINTS)}")
print(f"Working directory: {WORKDIR}")

## 2. Search & Download

Download monthly OPERA RTC-S1 passes. Both ascending and descending passes
are included — we will separate them later for analysis.

In [ ]:
from rs_tools.datasets.loader import load_dataset

data = load_dataset(
    "OPERA_RTC_S1",
    bbox=bbox_bypass,
    start_date=START_DATE,
    end_date=END_DATE,
    archive="terrascope",
    limit=500,
    monthly=True,
    interval_months=INTERVAL_MONTHS,
    output_dir=WORKDIR,
)
print(f"\n→ {len(data)} passes saved to {WORKDIR}/passes/")

## 3. Reload & Split by Orbit Direction

In [ ]:
data = load_passes_from_disk(WORKDIR)
print(f"{len(data)} passes available on disk\n")

# Split by orbit direction
asc_data  = [d for d in data if (d.orbit_direction or "").lower().startswith("asc")]
desc_data = [d for d in data if (d.orbit_direction or "").lower().startswith("desc")]
all_data  = data

print(f"  Ascending:  {len(asc_data)} passes")
print(f"  Descending: {len(desc_data)} passes")
print(f"  Total:      {len(all_data)} passes")

for item in data:
    orb = (item.orbit_direction or "?")[:3].upper()
    print(f"  {item.label}  orbit={orb}")

## 4. Acquisition Timeline by Orbit Direction

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
for item in asc_data:
    ax.scatter(item.datetime, 0.1, marker="|", s=300, c="steelblue", linewidths=2)
for item in desc_data:
    ax.scatter(item.datetime, -0.1, marker="|", s=300, c="darkorange", linewidths=2)
ax.scatter([], [], marker="|", s=100, c="steelblue", linewidths=2, label=f"Ascending ({len(asc_data)})")
ax.scatter([], [], marker="|", s=100, c="darkorange", linewidths=2, label=f"Descending ({len(desc_data)})")
ax.set_yticks([0.1, -0.1])
ax.set_yticklabels(["ASC", "DESC"])
ax.set_ylim(-0.4, 0.4)
ax.set_title("OPERA RTC-S1 acquisition timeline — Antwerp bypass (by orbit direction)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5. Backscatter Time-Series — ASC vs DESC vs Combined

For each point on the bypass, extract VV and VH backscatter (dB) from
ascending-only, descending-only, and combined passes.  If the **combined**
series shows more oscillation than the individual orbits, the effect is
geometric.

In [ ]:
def backscatter_db(items, point=None, crs="EPSG:4326"):
    """Extract backscatter in dB at a point or as AOI mean."""
    dates, vv_db, vh_db, sensors, orbits = [], [], [], [], []
    for item in items:
        item.load()
        vv_arr = item.data["VV"]
        vh_arr = item.data["VH"]
        if point is not None:
            raster_crs = vv_arr.rio.crs
            if raster_crs and str(raster_crs) != crs:
                transformer = pyproj.Transformer.from_crs(crs, str(raster_crs), always_xy=True)
                px, py = transformer.transform(point[0], point[1])
            else:
                px, py = point
            vv_val = float(vv_arr.sel(x=px, y=py, method="nearest").values)
            vh_val = float(vh_arr.sel(x=px, y=py, method="nearest").values)
        else:
            vv = vv_arr.values
            vh = vh_arr.values
            vv_val = float(np.nanmean(vv[vv > 0]))
            vh_val = float(np.nanmean(vh[vh > 0]))
        item.unload()
        if vv_val > 0 and vh_val > 0:
            dates.append(item.datetime)
            vv_db.append(10 * np.log10(vv_val))
            vh_db.append(10 * np.log10(vh_val))
            sensors.append(item.platform.split("-")[1] if "-" in item.platform else item.platform)
            orbits.append(item.orbit_direction or "unknown")
    return dates, vv_db, vh_db, sensors, orbits

In [ ]:
for point_name, point_coords in BYPASS_POINTS.items():
    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

    datasets = [
        ("Ascending only",  asc_data,  "steelblue"),
        ("Descending only", desc_data, "darkorange"),
        ("Combined",        all_data,  "forestgreen"),
    ]

    for ax, (label, subset, color) in zip(axes, datasets):
        if not subset:
            ax.set_title(f"{label} — no data")
            continue
        dates, vv, vh, sensors, orbits = backscatter_db(subset, point=point_coords)
        ax.plot(dates, vv, "o-", color=color, ms=4, lw=1.0, alpha=0.8, label="VV (dB)")
        ax.plot(dates, vh, "s-", color=color, ms=4, lw=1.0, alpha=0.5, label="VH (dB)")
        ax.set_ylabel("Backscatter (dB)")
        ax.set_title(f"{label} — {point_name}")
        ax.legend(loc="upper right", fontsize=8)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("Acquisition date")
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    fig.suptitle(f"Bypass oscillation — {point_name} ({point_coords[0]:.4f}, {point_coords[1]:.4f})",
                 fontsize=13)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

## 6. Oscillation Strength — Standard Deviation Comparison

If the standard deviation of the **combined** series is larger than the
individual orbit series, the oscillation is primarily due to mixing
viewing geometries.

In [ ]:
print(f"{'Point':<25s}  {'ASC σ(VV)':>10s}  {'DESC σ(VV)':>10s}  {'Combined σ(VV)':>14s}  {'Δ(Comb−max)':>12s}")
print("-" * 80)

for point_name, point_coords in BYPASS_POINTS.items():
    stds = {}
    for label, subset in [("ASC", asc_data), ("DESC", desc_data), ("Combined", all_data)]:
        if not subset:
            stds[label] = float("nan")
            continue
        _, vv, _, _, _ = backscatter_db(subset, point=point_coords)
        stds[label] = np.std(vv) if vv else float("nan")
    delta = stds["Combined"] - max(stds["ASC"], stds["DESC"])
    marker = " ← oscillation" if delta > 0.5 else ""
    print(f"{point_name:<25s}  {stds['ASC']:>10.3f}  {stds['DESC']:>10.3f}  "
          f"{stds['Combined']:>14.3f}  {delta:>+12.3f}{marker}")

## 7. RTC Composites — Ascending vs Descending

Side-by-side comparison of the latest ascending and descending passes to
highlight the geometric differences in how SAR illuminates the bypass
structures.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, subset, label in [(axes[0], asc_data, "Ascending"), (axes[1], desc_data, "Descending")]:
    if not subset:
        ax.set_title(f"{label} — no data")
        continue
    item = subset[-1]
    item.load()
    vv = item.data["VV"].values
    vh = item.data["VH"].values
    rgb = rtc_composite(vv, vh)
    ax.imshow(rgb, origin="upper")
    ax.set_axis_off()
    ax.set_title(f"{label} — {item.label}", fontsize=11)
    # Annotate bypass points
    for pname, pcoords in BYPASS_POINTS.items():
        try:
            annotate_location(ax, pcoords[0], pcoords[1], pname, item.data["VV"],
                              color="cyan", fontsize=7, markersize=3)
        except Exception:
            pass
    if item.pixel_size_m:
        add_scalebar(ax, item.pixel_size_m)
    item.unload()

plt.suptitle("Antwerp Bypass — Ascending vs Descending viewing geometry", fontsize=13)
plt.tight_layout()
plt.show()

## 8. Animated GIFs — Separate ASC, DESC, and Combined

Three GIFs to visually compare the temporal evolution from each viewing geometry.

In [ ]:
def _rtc_composite_fn(item):
    item.load()
    vv = item.data["VV"].values
    vh = item.data["VH"].values
    rgb = rtc_composite(vv, vh)
    label = item.label
    item.unload()
    return rgb, label

for subset, name in [(asc_data, "ascending"), (desc_data, "descending"), (all_data, "combined")]:
    if not subset:
        print(f"Skipping {name} GIF — no data")
        continue
    gif_path = os.path.join(GIF_DIR, f"bypass_{name}.gif")
    save_timeseries_gif_lazy(
        subset, gif_path,
        composite_fn=_rtc_composite_fn,
        title=f"Antwerp Bypass — {name.title()}",
        pixel_size_m=subset[0].pixel_size_m if subset[0].pixel_size_m else None,
        fps=2,
    )

## 9. Display GIFs

In [ ]:
from IPython.display import Image, display, Markdown

for name in ["ascending", "descending", "combined"]:
    gif_path = os.path.join(GIF_DIR, f"bypass_{name}.gif")
    if os.path.exists(gif_path):
        display(Markdown(f"### {name.title()}"))
        display(Image(filename=gif_path))

## 10. Summary & Conclusions

Look at the time-series plots above and the standard deviation table:

- If **ASC-only** and **DESC-only** have lower σ than **Combined**, the bypass
  oscillation is primarily a **viewing-geometry artefact** from mixing ascending
  and descending passes.
- If the oscillation persists even within a single orbit direction, it may
  reflect **real physical changes** (construction activity, surface roughness).